# Karpathy Tarzı Mini LLM

Bu not defteri, `microgpt.py` akışından ilham alan küçük bir karakter-seviyeli GPT'yi adım adım kurar.

Amaç:
- kendi ürettiğimiz coğrafya metinleri ile çalışmak,
- tokenizasyonu görmek,
- tek bir örnek üzerinde hesapların nasıl aktığını izlemek,
- sonra modeli eğitip yeni metin üretmek.

## Bu Not Defterinde 3 Şey

1. Türkçe coğrafya metinlerinden oluşan bir korpus kuruyoruz.
2. Karakter seviyesinde tokenizasyon ve model içi hesapları adım adım izliyoruz.
3. Küçük GPT'yi eğitip Türkçe bir başlangıç cümlesini tamamlatıyoruz.

## Adım Planı

1. Coğrafya odaklı kendi metin korpusumuzu oluşturacağız.
2. Karakter-seviyesinde tokenizasyon kuracağız.
3. Karpathy'nin microgpt akışındaki gibi `Value` tabanlı autograd yazacağız.
4. GPT benzeri küçük modeli tanımlayacağız: embedding, attention, MLP, softmax.
5. Tek bir örnek üzerinde ara hesapları yazdıracağız.
6. Korpusu eğitip kaybın nasıl azaldığını takip edeceğiz.
7. Sonunda yeni coğrafya metni üreteceğiz.

## Coğrafya Veri Seti

Aşağıda kendi ürettiğimiz, coğrafya anlatan kısa ama anlamlı bir metin korpusu hazırlıyoruz. Bu metinler daha sonra karakter-seviyeli tokenlara çevrilecek ve model bu metinleri taklit etmeyi öğrenecek.

In [2]:
from pathlib import Path
import random

random.seed(42)

corpus_docs = [
    "And Dağları, Güney Amerika'nın batı kenarı boyunca uzanan uzun bir dağ omurgası oluşturur. Yüksek geçitler, volkanik zirveler ve kuru platolar bölgenin iklimini şekillendirir.",
    "Amazon Havzası, nemli ormanlar, taşkın ovaları ve yavaş akan nehirlerle kaplıdır. Yağış, uzak yerleşimleri denize bağlayan yoğun bir su yolu ağı besler.",
    "Sahra, kum tepeleri, çakıl ovaları ve kayalık sırtlardan oluşan geniş bir çöldür. Kervan yolları, kuyuları, yıldızları ve rüzgârın ritmini izleyerek içinden geçerdi.",
    "Nil Nehri, geniş bir deltaya ulaşmadan önce dar bir vadi boyunca kuzeye akar. Su başka yerlerde kıt olduğu için çiftlikler, kasabalar ve sulama kanalları nehir çevresinde yoğunlaşır.",
    "Himalayalar, donmuş sırtlarını muson bulutlarının üstüne kaldırır. Kar örtüsü, buzullar ve derin vadiler, aşağı havzadaki milyonlarca insan için mevsimsel su depolar.",
    "Akdeniz kıyıları, limanları, adaları, burunları ve korunaklı koyları bir araya getirir. Ticaret, denizcilik ve iklim, kıyılarını binlerce yıldır birbirine bağlamıştır.",
    "Büyük Rift Vadisi, bir kıtayı ayıran tektonik kuvvetleri ortaya çıkarır. Göller, dik yamaçlar ve volkanik koniler hareket eden kabuğun izlerini taşır.",
    "Arktik Okyanusu, buz sahanlıkları, tundra ve eski buzulların oyduğu fiyortlarla çevrilidir. Mevsimsel karanlık ve düşük sıcaklıklar kuzeye yapılan her yolculuğu biçimlendirir.",
    "Pasifik adaları, geniş bir okyanus üzerinde serpiştirilmiş basamak taşları gibidir. Mercan resifleri, volkanik koniler ve lagün sistemleri kırılgan ekosistemler oluşturur.",
    "Orta Avrupa ovaları nehirler, demir yolu hatları ve verimli tarlalar tarafından kesilir. Ulaşım ve tarımın desteklediği yoğun nüfus, kavşaklara yakın kentler oluşturur.",
    "Colorado Nehri, yükselmiş kaya katmanları arasında derin kanyonlar oyar. İzlediği yol, erozyonun, yükselmenin ve zamanın bir platoyu nasıl uçurumlarla dolu bir manzaraya dönüştürdüğünü gösterir.",
    "Güney Asya'nın muson iklimi, yağışlı yazlar ve kuru kışlar getirir. Pirinç terasları, nehir deltaları ve kıyı limanları bu mevsimsel düzene uyum sağlar."
]

input_path = Path('/home/onc/workspace/dersler/2526/ileri_programlama/hafta13/input.txt')
input_path.write_text('\n\n'.join(corpus_docs), encoding='utf-8')

docs = [line.strip() for line in input_path.read_text(encoding='utf-8').split('\n\n') if line.strip()]
print(f'Doküman sayısı: {len(docs)}')
print('İlk doküman önizlemesi:')
print(docs[0])
print('\nKaydedilen korpus dosyası:', input_path)
print('Toplam karakter sayısı:', sum(len(doc) for doc in docs))

Doküman sayısı: 12
İlk doküman önizlemesi:
And Dağları, Güney Amerika'nın batı kenarı boyunca uzanan uzun bir dağ omurgası oluşturur. Yüksek geçitler, volkanik zirveler ve kuru platolar bölgenin iklimini şekillendirir.

Kaydedilen korpus dosyası: /home/onc/workspace/dersler/2526/ileri_programlama/hafta13/input.txt
Toplam karakter sayısı: 2017


## Tokenizasyon

Karpathy akışındaki gibi karakter-seviyesinde çalışacağız. Bu sayede her sembol tek bir token olur. `BOS` tokenı, dizinin başlangıcını işaret eder. Böylece model hangi karakter dizisinin başında olduğunu öğrenebilir.

In [3]:
docs = [line.strip() for line in input_path.read_text(encoding='utf-8').split('\n\n') if line.strip()]
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1
stoi = {ch: i for i, ch in enumerate(uchars)}
itos = {i: ch for ch, i in stoi.items()}
itos[BOS] = '<BOS>'


def encode(text):
    return [BOS] + [stoi[ch] for ch in text] + [BOS]


def decode(tokens):
    chars = []
    for token in tokens:
        if token == BOS:
            continue
        chars.append(itos[token])
    return ''.join(chars)

print('Benzersiz karakter sayısı:', len(uchars))
print('BOS dahil sözlük boyutu:', vocab_size)
print('Sözlükteki ilk 40 karakter:')
print(''.join(uchars[:40]))

example_doc = docs[0]
example_tokens = encode(example_doc)
print('\nÖrnek doküman:')
print(example_doc)
print('\nKodlanmış token idleri:')
print(example_tokens[:80])
print('\nTokenlardan geri çözüm:')
print(decode(example_tokens[:80]))

Benzersiz karakter sayısı: 51
BOS dahil sözlük boyutu: 52
Sözlükteki ilk 40 karakter:
 ',.ABCDGHKMNOPRSTUVYabcdefghiklmnoprstu

Örnek doküman:
And Dağları, Güney Amerika'nın batı kenarı boyunca uzanan uzun bir dağ omurgası oluşturur. Yüksek geçitler, volkanik zirveler ve kuru platolar bölgenin iklimini şekillendirir.

Kodlanmış token idleri:
[51, 4, 33, 24, 0, 7, 21, 47, 31, 21, 36, 49, 2, 0, 8, 46, 33, 25, 41, 0, 4, 32, 25, 36, 29, 30, 21, 1, 33, 49, 33, 0, 22, 21, 38, 49, 0, 30, 25, 33, 21, 36, 49, 0, 22, 34, 41, 39, 33, 23, 21, 0, 39, 42, 21, 33, 21, 33, 0, 39, 42, 39, 33, 0, 22, 29, 36, 0, 24, 21, 47, 0, 34, 32, 39, 36, 27, 21, 37, 49]

Tokenlardan geri çözüm:
And Dağları, Güney Amerika'nın batı kenarı boyunca uzanan uzun bir dağ omurgası


## Autograd ve Model

Bu hücrede Karpathy'nin microgpt yaklaşımındaki temel blokları kuracağız:
- `Value`: tek bir skalerin ileri ve geri geçiş bilgisini taşır.
- `linear`: ağırlıklı toplam.
- `softmax`: olasılığa dönüştürme.
- `rmsnorm`: vektörü ölçekleme.
- `gpt`: token + position embedding, attention, MLP ve son çıkış katmanı.

Sonra tek bir örnek üstünde bu parçaların nasıl işlediğini göstereceğiz.

In [4]:
import math
import random

random.seed(42)


class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = float(data)
        self.grad = 0.0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1.0, 1.0))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data**other, (self,), (other * self.data**(other - 1),))

    def log(self):
        return Value(math.log(self.data), (self,), (1.0 / self.data,))

    def exp(self):
        value = math.exp(self.data)
        return Value(value, (self,), (value,))

    def relu(self):
        return Value(max(0.0, self.data), (self,), (1.0 if self.data > 0 else 0.0,))

    def __neg__(self):
        return self * -1.0

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._children:
                    build_topo(child)
                topo.append(node)

        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            for child, local_grad in zip(node._children, node._local_grads):
                child.grad += local_grad * node.grad


def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(row, x)) for row in w]


def softmax(logits):
    max_val = max(value.data for value in logits)
    exps = [(value - max_val).exp() for value in logits]
    total = sum(exps)
    return [value / total for value in exps]


def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]


def matrix(nout, nin, std=0.08):
    return [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]


n_layer = 1
n_embd = 12
block_size = 24
n_head = 3
head_dim = n_embd // n_head

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'wpe': matrix(block_size, n_embd),
    'lm_head': matrix(vocab_size, n_embd),
}
for layer_index in range(n_layer):
    state_dict[f'layer{layer_index}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{layer_index}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{layer_index}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{layer_index}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{layer_index}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{layer_index}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)

params = [value for matrix_ in state_dict.values() for row in matrix_ for value in row]
print('Parametre sayısı:', len(params))


def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]
    x = rmsnorm(x)

    for layer_index in range(n_layer):
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{layer_index}.attn_wq'])
        k = linear(x, state_dict[f'layer{layer_index}.attn_wk'])
        v = linear(x, state_dict[f'layer{layer_index}.attn_wv'])
        keys[layer_index].append(k)
        values[layer_index].append(v)

        attn_out = []
        for head_index in range(n_head):
            start = head_index * head_dim
            q_h = q[start:start + head_dim]
            k_h = [entry[start:start + head_dim] for entry in keys[layer_index]]
            v_h = [entry[start:start + head_dim] for entry in values[layer_index]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            attn_out.extend(head_out)

        x = linear(attn_out, state_dict[f'layer{layer_index}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]

        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{layer_index}.mlp_fc1'])
        x = [value.relu() for value in x]
        x = linear(x, state_dict[f'layer{layer_index}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits


def zero_grads():
    for value in params:
        value.grad = 0.0


def loss_for_doc(doc):
    tokens = encode(doc)
    n = min(block_size, len(tokens) - 1)
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1.0 / n) * sum(losses)
    return loss, tokens, n


def inspect_one_document(doc, positions=3):
    tokens = encode(doc)
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]
    print('Doküman önizlemesi:')
    print(doc)
    print('\nToken idleri:')
    print(tokens[:positions + 1])
    for pos_id in range(min(positions, len(tokens) - 1)):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        print(f'\nPozisyon {pos_id}')
        print('Mevcut token:', token_id, repr(itos[token_id]))
        print('Hedef token  :', target_id, repr(itos[target_id]))
        print('İlk 8 logit:')
        print([round(value.data, 4) for value in logits[:8]])
        print('İlk 8 olasılık:')
        print([round(value.data, 4) for value in probs[:8]])
        print('Hedef olasılığı:', round(probs[target_id].data, 6))
        print('Adım kaybı:', round((-probs[target_id].log()).data, 6))


inspect_one_document(docs[0], positions=3)

Parametre sayısı: 3264
Doküman önizlemesi:
And Dağları, Güney Amerika'nın batı kenarı boyunca uzanan uzun bir dağ omurgası oluşturur. Yüksek geçitler, volkanik zirveler ve kuru platolar bölgenin iklimini şekillendirir.

Token idleri:
[51, 4, 33, 24]

Pozisyon 0
Mevcut token: 51 '<BOS>'
Hedef token  : 4 'A'
İlk 8 logit:
[-0.1217, -0.0077, 0.5606, -0.6911, 0.1277, -0.1413, 0.1988, -0.2167]
İlk 8 olasılık:
[0.0161, 0.018, 0.0318, 0.0091, 0.0206, 0.0157, 0.0221, 0.0146]
Hedef olasılığı: 0.020604
Adım kaybı: 3.88226

Pozisyon 1
Mevcut token: 4 'A'
Hedef token  : 33 'n'
İlk 8 logit:
[-0.4132, -0.2455, 0.0805, -0.0783, -0.3776, 0.8131, -0.1433, 0.3118]
İlk 8 olasılık:
[0.0126, 0.0149, 0.0207, 0.0176, 0.0131, 0.043, 0.0165, 0.026]
Hedef olasılığı: 0.025542
Adım kaybı: 3.66743

Pozisyon 2
Mevcut token: 33 'n'
Hedef token  : 24 'd'
İlk 8 logit:
[-0.0878, -0.2497, 0.3762, -0.0582, -0.3109, 0.1254, 0.0247, 0.5145]
İlk 8 olasılık:
[0.0176, 0.0149, 0.0279, 0.0181, 0.014, 0.0217, 0.0196, 0.0321]
Hede

## Eğitim Döngüsü

Bu aşamada her dokümanı sırayla modelden geçirip ortalama loss hesaplayacağız. Sonra `loss.backward()` ile gradyanları çıkarıp Adam ile parametreleri güncelleyeceğiz. Böylece Karpathy'nin kodundaki "forward -> loss -> backward -> update" zincirini açıkça görebileceğiz.

In [6]:
class AdamOpt:
    def __init__(self, size, lr=1e-3, beta1=0.85, beta2=0.99, eps=1e-8):
        self.m = [0.0] * size
        self.v = [0.0] * size
        self.t = 0
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps

    def step(self, params, grads):
        self.t += 1
        new_params = []
        for i, (param, grad) in enumerate(zip(params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (grad * grad)
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            new_params.append(param - self.lr * m_hat / (v_hat ** 0.5 + self.eps))
        return new_params


def train_step(doc):
    tokens = encode(doc)
    n = min(block_size, len(tokens) - 1)
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        losses.append(-probs[target_id].log())
    loss = (1.0 / n) * sum(losses)
    loss.backward()
    return loss


def flatten_grads(param_list):
    return [param.grad for param in param_list]


def reset_param_grads(param_list):
    for param in param_list:
        param.grad = 0.0


def set_param_values(param_list, new_values):
    for param, new_value in zip(param_list, new_values):
        param.data = float(new_value)


learning_rate = 0.01
optimizer = AdamOpt(len(params), lr=learning_rate, beta1=0.85, beta2=0.99)
training_steps = 120
loss_history = []

print('Eğitim başladı...')
for step in range(training_steps):
    doc = docs[step % len(docs)]
    loss = train_step(doc)
    grads = flatten_grads(params)
    new_param_values = optimizer.step([param.data for param in params], grads)
    set_param_values(params, new_param_values)
    reset_param_grads(params)
    loss_history.append(loss.data)
    if step % 5 == 0 or step == training_steps - 1:
        print(f'adım {step + 1:02d}/{training_steps} | loss = {loss.data:.4f}')

print('\nLoss geçmişi:')
print([round(value, 4) for value in loss_history])

Eğitim başladı...
adım 01/120 | loss = 2.3915
adım 06/120 | loss = 2.4920
adım 11/120 | loss = 2.8675
adım 16/120 | loss = 2.1225
adım 21/120 | loss = 2.2707
adım 26/120 | loss = 2.0911
adım 31/120 | loss = 2.0764
adım 36/120 | loss = 2.3816
adım 41/120 | loss = 1.7738
adım 46/120 | loss = 2.4657
adım 51/120 | loss = 1.6344
adım 56/120 | loss = 1.9183
adım 61/120 | loss = 1.2853
adım 66/120 | loss = 1.4948
adım 71/120 | loss = 1.7532
adım 76/120 | loss = 1.1620
adım 81/120 | loss = 1.2787
adım 86/120 | loss = 1.3034
adım 91/120 | loss = 1.1320
adım 96/120 | loss = 1.6673
adım 101/120 | loss = 1.0785
adım 106/120 | loss = 1.4024
adım 111/120 | loss = 1.0056
adım 116/120 | loss = 1.5505
adım 120/120 | loss = 1.5231

Loss geçmişi:
[2.3915, 2.5906, 2.5082, 2.3392, 2.8287, 2.492, 3.0188, 3.1489, 2.6432, 3.2353, 2.8675, 3.1179, 1.8841, 2.2746, 2.2378, 2.1225, 2.12, 2.0312, 2.414, 2.5633, 2.2707, 2.9999, 2.5662, 2.6327, 1.6214, 2.0911, 1.9788, 1.816, 1.9516, 1.8241, 2.0764, 2.2451, 1.9757, 2.

## Metin Üretimi

Eğitimden sonra modele bir coğrafya başlangıcı verip devamını örnekleyeceğiz. Bu, `microgpt.py` içindeki inference bölümünün notebook karşılığıdır.

In [7]:
def sample_next_token(logits, temperature=0.8):
    scaled = [value / temperature for value in logits]
    probs = softmax(scaled)
    return random.choices(range(vocab_size), weights=[value.data for value in probs])[0]


def generate_text(prompt, max_new_chars=140, temperature=0.8):
    tokens = encode(prompt)
    generated = tokens[:]
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]

    for pos_id, token_id in enumerate(tokens[:-1]):
        _ = gpt(token_id, pos_id, keys, values)

    token_id = tokens[-1]
    for pos_id in range(len(tokens) - 1, min(block_size, len(tokens) - 1) + max_new_chars):
        logits = gpt(token_id, min(pos_id, block_size - 1), keys, values)
        token_id = sample_next_token(logits, temperature=temperature)
        if token_id == BOS:
            break
        generated.append(token_id)

    return decode(generated)


prompt = 'Nehir '
print('Başlangıç metni:')
print(prompt)
print('\nÜretilen devam:')
print(generate_text(prompt, max_new_chars=120, temperature=0.7))

Başlangıç metni:
Nehir 

Üretilen devam:
Nehir ükyavoni, gel deliklikliklililiklikliklikliklikklaliklikkliksikkkiklikkklikkkklikliklolkkklkkkkkkkkkkklikktikikkkkkkkkkk


## Eğitim ve Kayıp Hesabı

Her eğitim adımında sıralı bir coğrafya dokümanı alacağız. Tokenlar modelden geçecek, hedef karakter ile tahmin arasındaki fark `loss` olarak ölçülecek ve ardından `backward()` ile gradyanlar hesaplanacak. Sonra Adam güncellemesi ile ağırlıklar değiştirilecek.